In [ ]:
import pandas as pd
import numpy as 

## Data Cleaning

In [2]:
df = pd.read_csv('egypt_real_estate_listings.csv')
features = ['price', 'location', 'type', 'size', 'bedrooms', 'bathrooms']

df_model = df[features]

# Remove entries without a price
df_model = df_model.dropna(subset=['price'], axis=0)
# Remove entries without bedroom values
df_model = df_model.dropna(subset=['bedrooms', 'bathrooms'], axis=0)

In [3]:
# Removing commas from 'price'
df_model['price'] = df_model['price'].str.replace(',', '').astype(float)

In [4]:
# Renaming 'price' to 'price_egp'
# Adding 'price_usd'
pd.set_option('display.float_format', '{:.2f}'.format)
df_model['price_usd'] = df_model['price'] * 0.210172

In [5]:
# Ensuring numeric columns are numeric
# Replacing '7+' values to numerical value 7
df_model['bathrooms'] = df_model['bathrooms'].replace('7+', 7)
df_model = df_model[df_model['bathrooms'] != 'none']
df_model['bathrooms'] = df_model['bathrooms'].astype(float)
df_model['bathrooms'] = df_model['bathrooms'].astype(int)

In [6]:
# Create binary column whether or not house has a maid room 
# (0: no maid room, 1: has maid room)
df_model['maid_room'] = df_model['bedrooms'].str.contains('Maid', case=False, na=False)

In [7]:
# Replace 'studio' with '0', 'studio+ Maid' with '0+ Maid'
df_model['bedrooms'] = df_model['bedrooms'].replace({'studio': '0', 'studio+ Maid': '0+ Maid'})
df_model['bedrooms'] = df_model['bedrooms'].str.split('+').str[0]
df_model['bedrooms'] = df_model['bedrooms'].astype(int)
print('Uniqe bedrooms col: ', df_model['bedrooms'].unique())

Uniqe bedrooms col:  [1 4 2 3 7 5 0 6]


In [8]:
# Only including entries with 'sqft' in 'size' column
# some entries includes sqm 
df_model = df_model[df_model['size'].str.contains('sqft')]

In [9]:
# Only include square footage as it also includes the square meter
df_model['size'] = df_model['size'].str.split('sqft').str[0]

df_model['size'] = df_model['size'].str.replace(',', '')
df_model['size'] = df_model['size'].astype(int)

In [10]:
# dropping outliers based on price
df_model.drop([4410, 15510, 8174], inplace=True)
df_model.drop([14334, 2693, 14774], inplace=True)

outlier = df_model[df_model['price'] == 840000000].index
df_model.drop(index=outlier, inplace=True)

outlier = df_model[df_model['price'] == 650000000].index
df_model.drop(index=outlier, inplace=True)

In [11]:
# limit the listings to under 5000 square feet to be realistic 
df_model = df_model[df_model['size'] <= 5000]

## Creating the Models 
### Comparing Random Forest Regression and Linear Regression

In [12]:
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression